# Analisi del Sentiment con Hugging Face e CI/CD

## WHY - Perché questo progetto?

L'analisi del sentiment è un compito fondamentale di Natural Language Processing (NLP) con applicazioni reali:
- **Monitoraggio del brand**: tracciare come i clienti parlano del tuo prodotto
- **Customer support**: prioritizzare ticket negativi per risposta rapida
- **Social listening**: comprendere il sentiment pubblico su argomenti specifici

Questo progetto dimostra come **implementare una soluzione ML-Ops completa**: non solo un modello, ma un sistema produttivo con CI/CD, testing automatico, deployment e monitoring continuo.

## WHAT - Cosa farai in questo notebook?

- **Setup**: configurare l'ambiente con dipendenze e pacchetto Python
- **Dataset**: caricare e esplorare tweet etichettati con sentiment (negativo/neutrale/positivo)
- **Inference**: testare il modello pre-addestrato `cardiffnlp/twitter-roberta-base-sentiment-latest`
- **Testing**: eseguire test automatici per validare il modello
- **Monitoring**: valutare performance su dati reali
- **Deploy**: capire come pubblicare il modello su Hugging Face Hub

## HOW - Come raggiungeremo questi obiettivi?

Uso di **framework e best practices moderne**:
- **Hugging Face Transformers**: libreria standard per NLP pre-addestrato
- **Hugging Face Datasets**: caricamento efficiente di dataset pubblici
- **GitHub Actions**: CI/CD pipeline automatica
- **Python packaging**: struttura professionale con `pyproject.toml` e `setup.py`
- **Pytest**: testing automatico

Link al repository GitHub: https://github.com/emanuelesalbego-droid/ML-Ops-e-machine-learning-in-operazione

## 1. Setup e installazione

### WHY - Perché è importante il setup corretto?

Un ambiente Python correttamente configurato è **essenziale per la riproducibilità**:
- Le stesse versioni di librerie garantiscono risultati coerenti
- Evita conflitti di dipendenze (es. PyTorch con/senza GPU)
- Il packaging rende il codice riutilizzabile e distribuibile

### WHAT - Cosa installiamo?

- **pip**: gestore di pacchetti Python aggiornato
- **Pacchetto locale**: installa il codice del progetto in modalità sviluppo (`-e`)
- **Dipendenze produzione**: librerie richieste da `requirements.txt` (transformers, torch, datasets, scikit-learn, etc.)

### HOW - Come facciamo?

L'esecuzione della cella seguente:
1. Aggiorna pip alla versione più recente
2. Installa il progetto locale in **modalità editable** (i cambiamenti al codice sono immediatamente disponibili)
3. Installa tutte le dipendenze necessarie dal file `requirements.txt`

**Nota**: la prima esecuzione potrebbe richiedere alcuni minuti per scaricare i modelli pre-addestrati (vengono salvati in cache locale).

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -e /workspaces/ML-Ops-e-machine-learning-in-operazione
!python -m pip install -r /workspaces/ML-Ops-e-machine-learning-in-operazione/requirements.txt

### Risultato atteso

Dopo l'installazione, avrai:
- ✅ Accesso ai moduli `src.data`, `src.model`, `src.predict`, etc. tramite import diretto
- ✅ Tutte le dipendenze necessarie per NLP con Hugging Face
- ✅ Pytorch configurato per CPU (o GPU se disponibile)
- ✅ Librerie di testing (pytest) e utilities (pandas, scikit-learn)

## 2. Caricamento del dataset e panoramica

### WHY - Perché il dataset `tweet_eval`?

- **Realismo**: tweet reali da Twitter con linguaggio informale, emoji, errori di ortografia
- **Valutazione pre-addestrata**: il modello `cardiffnlp/twitter-roberta-base-sentiment-latest` è stato **fine-tuned proprio su questo dataset**
- **Pubblicamente disponibile**: accesso via Hugging Face Datasets Hub, nessuna configurazione manuale di credenziali
- **Equilibrato**: distribuito ragionevolmente tra le 3 classi (negativo, neutrale, positivo)

### WHAT - Cosa contiene il dataset?

Ogni riga del dataset `tweet_eval` contiene:
- `text`: il testo del tweet (50-280 caratteri)
- `label`: etichetta numerica 0=negativo, 1=neutrale, 2=positivo
- Split disponibili: 'train' (~30k), 'validation', 'test'

### HOW - Come lo usiamo?

La cella seguente:
1. Carica il dataset dalla cache locale (o scarica da Hub la prima volta)
2. Seleziona solo i primi 20 campioni per visualizzazione veloce
3. Mostra la struttura: colonne, tipi di dato, campioni

In [ ]:
from datasets import load_dataset

dataset = load_dataset('tweet_eval', 'sentiment', split='validation')
dataset = dataset.select(range(20))
dataset

### Osservazioni sulla struttura

- Colonna `text`: testo grezzo del tweet (non pulito, può contenere menzioni @, hashtag, URL)
- Colonna `label`: intero da 0 a 2 (mapping: 0→negativo, 1→neutrale, 2→positivo)
- Formato: oggetto `Dataset` di Hugging Face (simile a un DataFrame, ma ottimizzato per caricamento in streaming)

Questo è il formato che il modello di sentiment analysis si aspetta: **testo grezzo senza preprocessing manuale**, poiché il tokenizer interno si occupa di tutto.

## 3. Inferenza con il modello pre-addestrato

### WHY - Perché usare un modello pre-addestrato?

**Training da zero richiederebbe**:
- Migliaia di GPU-ore
- Decine di migliaia di etichette
- Expertise in tuning di iperparametri

**Pre-addestrato significa**:
- Modello già fine-tuned su Twitter data (dominio specifico)
- Pronto all'uso: basta installare e usare
- Accuratezza > 90% su questo task
- Eseguibile su CPU in tempo reale

### WHAT - Cosa fa il pipeline?

Il `pipeline` di Hugging Face:
1. **Tokenizza** il testo (converte in token IDs)
2. **Passa attraverso il modello** (RoBERTa fine-tuned)
3. **Post-processa** i logit in probabilità (softmax)
4. **Ritorna** una lista di dizionari con `label` e `score`

### HOW - Come eseguiamo l'inferenza?

La cella seguente:
1. Istanzia il pipeline con modello e tokenizer specifici
2. Passa 3 frasi di esempio (chiara progressione: positivo → neutrale → negativo)
3. Ritorna le predizioni con etichetta e score di confidenza

In [ ]:
from transformers import pipeline

sentiment = pipeline('sentiment-analysis', model='cardiffnlp/twitter-roberta-base-sentiment-latest', tokenizer='cardiffnlp/twitter-roberta-base-sentiment-latest')
samples = [
    'I love this new feature, it is fantastic!',
    'It is okay, not great but not terrible.',
    'This is the worst experience I have ever had.'
]
results = sentiment(samples)
results

### Interpretazione dei risultati

Ogni elemento dell'output contiene:
- `label`: etichetta predetta ('positive', 'negative', 'neutral')
- `score`: confidenza della predizione (0.0-1.0, dove 1.0 = massima confidenza)

**Esempio**: `{'label': 'positive', 'score': 0.95}` significa:
- Il modello classifica il testo come **positivo**
- Con confidenza del **95%**

### Quando è utile il score?

- **Score alto (>0.9)**: fiducia alta, predizione affidabile
- **Score medio (0.5-0.8)**: moderata incertezza, potrebbe richiedere intervento umano
- **Score basso (<0.5)**: bassa confidenza, testo ambiguo o fuori dominio

In sistemi di produzione, usare il score per:
- Routing automatico (high confidence → automatico, low → review manuale)
- Identificare errori (feedback loops per miglioramenti continui)

## 5. Deploy, monitoraggio e CI/CD

### WHY - Perché ML-Ops?

Il modello addestrato è solo il **5-10% del lavoro di produzione**:
- **Versioning**: quale versione del modello è in produzione?
- **Reproducibility**: è possibile riaddestrare con gli stessi risultati?
- **Monitoring**: le performance stanno degradando (data drift)?
- **Scale**: come gestire migliaia di predizioni/secondo?
- **Governance**: chi ha il diritto di deployare? Quali audit trail?

### WHAT - Cosa contiene il progetto?

**Script principali** (in `src/`):

| File | Scopo | Quando usarlo |
|------|-------|---|
| `train.py` | Fine-tuning del modello su dataset | Periodicamente per migliorare accuratezza |
| `predict.py` | Inferenza su testi nuovi | In produzione per classificare testi |
| `deploy.py` | Carica modello su Hugging Face Hub | Prima di usare il modello in altri servizi |
| `monitor.py` | Valuta performance su test set | Controllare degradazione (data drift) |

**Automazione**: `.github/workflows/ci.yml`
- Esegue automaticamente test ad ogni push
- Addestra il modello su subset di dati
- Deploya su Hugging Face (opzionale, richiede secrets)

### HOW - Come funziona il flusso end-to-end?

```
1. Developer commits codice
   ↓
2. GitHub Actions lancia CI workflow
   ├─ Installa dipendenze
   ├─ Esegue pytest (verifica che modello funziona)
   ├─ Addestra modello su subset (smoke test)
   └─ Deploya su Hub (se secrets configurati)
   ↓
3. Modello in produzione pronto per inferenza
   ↓
4. Monitoraggio continuo con src/monitor.py
   └─ Se performance degrada → alert → retraining
```

### Prossimi passi per il deploy

1. **Configura i segreti GitHub**:
   - `HF_TOKEN`: token di accesso a Hugging Face (genera su https://huggingface.co/settings/tokens)
   - `HF_MODEL_REPO`: username/model-name (es. emanuelesalbego-droid/sentiment-model)

2. **Triggera il deploy manuale**:
   - Vai in "Actions" → "CI/CD" → "Run workflow"

3. **Usa il modello da altri servizi**:
   ```python
   from transformers import pipeline
   sentiment = pipeline('sentiment-analysis', model='username/model-name')
   result = sentiment("Your text here")
   ```

4. **Monitoring in produzione**:
   ```bash
   python src/monitor.py --sample-size 512 --output production-metrics.json
   ```
   Genera report con precision/recall/f1 per identificare degradazione di performance.

## 6. Concetti avanzati: Data Drift e Model Degradation

### Il problema: Performance di modelli ML nel tempo

In produzione, i modelli ML non rimangono buoni per sempre:

**Data Drift**: i dati cambiano nel tempo
- Esempio: il linguaggio sui social media evolve (slang, trend, emoji nuove)
- Risultato: il modello addestrato ieri potrebbe essere inaccurato oggi

**Model Degradation**: l'accuratezza scende
- Se nel 2024 l'accuratezza era 92%, nel 2025 potrebbe essere 87%
- Segno che il modello ha bisogno di retraining

### Come lo riconosciamo?

Usiamo `src/monitor.py` per:
1. Campionare 512+ testi reali (dai dati di produzione)
2. Eseguire predizioni
3. Comparare con le etichette reali (ground truth)
4. Calcolare metriche:
   - **Accuracy**: % di predizioni corrette
   - **Precision**: di quelle predict positive, quante realmente lo sono?
   - **Recall**: dei veri positivi, quanti il modello ha trovato?
   - **F1-Score**: media armonica (useful per dataset sbilanciati)

### Azione consigliate

- **F1 > 0.85**: modello performante, continua normalmente
- **F1 0.70-0.85**: degradazione moderata, avvia retraining
- **F1 < 0.70**: degradazione severa, fallback a modulo alternativo

In un sistema ML-Ops completo, ciò è **automatizzato con alert e retraining trigger automatici**.

## 4. Testing automatico

### WHY - Perché testare un modello ML?

A differenza del software tradizionale:
- **Non c'è risposta "corretta"** univoca (il modello fa predizioni probabilistiche)
- **Serve accertarsi che il modello carica e funziona** senza errori
- **Regressioni**: cambiamenti al codice potrebbero romperlo
- **CI/CD inaffidabile senza test**: deploy cieco di modelli fallati in produzione

### WHAT - Cosa testiamo?

Nel file `tests/test_model.py`:
- ✅ Il modello carica senza errori
- ✅ L'inferenza ritorna una lista di dict (struttura corretta)
- ✅ Ogni dict contiene `label` e `score`
- ✅ L'etichetta è una delle 3 classi attese

### HOW - Come eseguiamo i test?



In [ ]:
import subprocess

# Esegui i test del progetto
result = subprocess.run(
    ["python", "-m", "pytest", "tests/test_model.py", "-v"],
    cwd="/workspaces/ML-Ops-e-machine-learning-in-operazione",
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

### Lettura dei risultati

- ✅ **PASSED**: il test è passato con successo
- ❌ **FAILED**: il test non ha passato (errore nel modello o nella logica)
- ⏭️ **SKIPPED**: il test è stato saltato (es. per environment specifico)

Se vedi 3 PASSED, il modello è pronto per l'uso in questa cella e continuerà a funzionare nei prossimi step.